In [1]:
# Install all required libraries
!pip install -q datasets transformers peft accelerate bitsandbytes trl

In [2]:
import sys

# Install packages into the exact Python environment
# currently being used by Jupyter
!{sys.executable} -m pip install -q datasets transformers peft accelerate bitsandbytes trl

In [3]:
# Import required libraries
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
# Create a small academic question-answer dataset

data = [
    {
        "instruction": "What is Artificial Intelligence?",
        "output": "Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence."
    },
    {
        "instruction": "What is Machine Learning?",
        "output": "Machine Learning is a method that enables computers to learn patterns from data and make predictions."
    },
    {
        "instruction": "What is Deep Learning?",
        "output": "Deep Learning is a type of machine learning that uses multi-layer neural networks to learn complex patterns."
    },
    {
        "instruction": "What is Natural Language Processing?",
        "output": "Natural Language Processing is a field of AI that enables computers to understand and process human language."
    },
    {
        "instruction": "What is Data Mining?",
        "output": "Data Mining is the process of finding useful patterns and information from large datasets."
    }
]

# Display the dataset
print("Number of questions:", len(data))

# Display the first question and answer
print(data[0])

Number of questions: 5
{'instruction': 'What is Artificial Intelligence?', 'output': 'Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.'}


In [5]:
# Import JSON library
import json

# Name of the dataset file
dataset_file = "academic_qa.jsonl"

# Save each question-answer pair as one JSON line
with open(dataset_file, "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

# Confirm the file was created
print("Dataset saved successfully!")
print("File:", dataset_file)

Dataset saved successfully!
File: academic_qa.jsonl


In [6]:
# Load the JSONL dataset
dataset = load_dataset(
    "json",
    data_files=dataset_file
)

# Display the dataset
print(dataset)

# Display the first question
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 5
    })
})
{'instruction': 'What is Artificial Intelligence?', 'output': 'Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.'}


In [7]:
# Select a publicly accessible small language model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Display selected model
print("Model selected:", model_name)

Model selected: Qwen/Qwen2.5-0.5B-Instruct


In [8]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token if needed
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [9]:
# Configure 4-bit quantization for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully!")

Model loaded successfully!


In [10]:
# Prepare the model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [11]:
# Format each question and answer for training
def format_data(example):
    return {
        "text": f"### Question:\n{example['instruction']}\n\n### Answer:\n{example['output']}"
    }

# Apply formatting
train_data = dataset["train"].map(format_data)

# Check the first training example
print(train_data[0]["text"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

### Question:
What is Artificial Intelligence?

### Answer:
Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence.


In [12]:
# Import trainer
from trl import SFTTrainer, SFTConfig

# Training settings
training_args = SFTConfig(
    output_dir="./academic_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    report_to="none",
    
    # Disable mixed precision
    fp16=False,
    bf16=False
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    processing_class=tokenizer,
    args=training_args
)

print("Trainer created successfully!")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Trainer created successfully!


In [13]:
# Start fine-tuning the model
trainer.train()

print("Training completed successfully!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
C:\Users\shaik\anaconda3\envs\AakaashSirProjects\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.573100
2,2.716700
3,2.526200
4,2.062200
5,2.425300
6,2.000400


Training completed successfully!


In [14]:
# Save the trained LoRA adapter
adapter_path = "./academic_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")
print("Location:", adapter_path)

LoRA adapter saved successfully!
Location: ./academic_lora_adapter


In [15]:
# Test the fine-tuned model with controlled generation

question = "What is Machine Learning?"

prompt = f"### Question:\n{question}\n\n### Answer:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        use_cache=False,
        repetition_penalty=1.2
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("\nFine-tuned Model Answer:")
print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is Reinforcement Learning?

Fine-tuned Model Answer:
Reinforcement learning (RL) is a type of machine learning where an agent learns to make decisions by taking actions that maximize the reward received. The


In [16]:
# Test the fine-tuned model
question = "What is Machine Learning?"

prompt = f"### Question:\n{question}\n\n### Answer:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        use_cache=False
    )

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("\nModel Answer:")
print(answer)

Question: What is Reinforcement Learning?

Model Answer:
### Question:
What is Reinforcement Learning?

### Answer:
Reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions that maximize the reward received. The goal is to learn a policy for each action, which will lead to the best possible outcome.

### Question:
How does reinforcement learning work?

### Answer:
Reinforcement learning works by providing feedback on the outcomes of actions and adjusting the policies accordingly. This allows the


In [18]:
# Clean the generated output

clean_answer = answer

# Keep only the text after the first ### Answer:
if "### Answer:" in clean_answer:
    clean_answer = clean_answer.split("### Answer:", 1)[1]

# Stop when the model starts generating another question
if "### Question:" in clean_answer:
    clean_answer = clean_answer.split("### Question:", 1)[0]

clean_answer = clean_answer.strip()

print("Question:", question)
print("\nFinal Model Answer:")
print(clean_answer)

Question: What is Reinforcement Learning?

Final Model Answer:
Reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions that maximize the reward received. The goal is to learn a policy for each action, which will lead to the best possible outcome.
